In [2]:
!nvidia-smi

Sun Sep 27 05:36:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!python --version

Python 3.13.15


In [4]:
!pip install -U unsloth

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
  Attempting uninstall: datasets
    Found existing installation: datasets 5.0.1
    Uninstalling datasets-5.0.1:
      Successfully uninstalled datasets-5.0.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.17.0
    Uninstalling tr

In [1]:
import unsloth

print("Unsloth installed successfully!")
print("Unsloth version:", unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully!
Unsloth version: 2026.9.11


In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset
import torch

print("All libraries imported successfully!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

All libraries imported successfully!
PyTorch version: 2.11.0+cu128
CUDA available: True


In [4]:
gpu_name = torch.cuda.get_device_name(0)

print("GPU:", gpu_name)
print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

GPU: Tesla T4
GPU Memory: 14.56 GB


In [5]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


In [6]:
import unsloth

print("Unsloth installed successfully!")
print("Unsloth version:", unsloth.__version__)

Unsloth installed successfully!
Unsloth version: 2026.9.11


In [7]:
!wget -q https://huggingface.co/datasets/casey-martin/MedInstruct/resolve/main/data/MedInstruct-52k.json -O MedInstruct-52k.json

In [8]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="MedInstruct-52k.json",
    split="train"
)

print(dataset)
print(dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 52002
})
['instruction', 'input', 'output']


In [9]:
print(dataset[0])

{'instruction': 'Explain why a mass in the lung could cause shortness of breath.', 'input': '<noinput>', 'output': 'A mass in the lung could cause shortness of breath due to several reasons. First, the mass can physically obstruct the air passages, causing difficulty in airflow and leading to breathing difficulties. Second, if the mass is cancerous or infected, it can cause inflammation and damage to lung tissue, reducing its functional capacity and compromising normal breathing. Additionally, a lung mass can compress adjacent structures such as blood vessels, bronchi, or the diaphragm, further impeding normal respiratory function. Overall, any interference with the normal flow of air in the lungs caused by a mass can result in inadequate oxygen exchange and subsequent shortness of breath.\n\nThe answer is: A mass in the lung can obstruct air passages, cause inflammation, damage lung tissue, and compress adjacent structures, leading to shortness of breath.'}


In [10]:
dataset = dataset.select_columns([
    "instruction",
    "input",
    "output"
])

print(dataset)
print(dataset.column_names)
print("Total examples:", len(dataset))

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 52002
})
['instruction', 'input', 'output']
Total examples: 52002


In [11]:
def format_medical_example(example):
    instruction = example["instruction"]
    user_input = example["input"]
    answer = example["output"]

    if user_input and str(user_input).strip():
        text = f"""### Instruction:
{instruction}

### Medical Question/Context:
{user_input}

### Answer:
{answer}"""
    else:
        text = f"""### Instruction:
{instruction}

### Answer:
{answer}"""

    return {"text": text}


medical_data = dataset.map(format_medical_example)

train_dataset = medical_data

print("Total training examples:", len(train_dataset))
print(train_dataset[0]["text"])

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

Total training examples: 52002
### Instruction:
Explain why a mass in the lung could cause shortness of breath.

### Medical Question/Context:
<noinput>

### Answer:
A mass in the lung could cause shortness of breath due to several reasons. First, the mass can physically obstruct the air passages, causing difficulty in airflow and leading to breathing difficulties. Second, if the mass is cancerous or infected, it can cause inflammation and damage to lung tissue, reducing its functional capacity and compromising normal breathing. Additionally, a lung mass can compress adjacent structures such as blood vessels, bronchi, or the diaphragm, further impeding normal respiratory function. Overall, any interference with the normal flow of air in the lungs caused by a mass can result in inadequate oxygen exchange and subsequent shortness of breath.

The answer is: A mass in the lung can obstruct air passages, cause inflammation, damage lung tissue, and compress adjacent structures, leading to sh

In [12]:
medical_data = dataset.map(format_medical_example)

In [13]:
def format_medical_example(example):
    instruction = example["instruction"]
    user_input = example["input"]
    answer = example["output"]

    if user_input and str(user_input).strip():
        text = f"""### Instruction:
{instruction}

### Medical Question/Context:
{user_input}

### Answer:
{answer}"""
    else:
        text = f"""### Instruction:
{instruction}

### Answer:
{answer}"""

    return {"text": text}


# Format EVERY example in the dataset
train_dataset = dataset.map(
    format_medical_example,
    desc="Formatting all medical questions"
)

print("Total questions used for training:", len(train_dataset))

Total questions used for training: 52002


In [14]:
with open("all_medical_training_data.txt", "w", encoding="utf-8") as f:

    for i, example in enumerate(train_dataset):

        f.write("=" * 100 + "\n")
        f.write(f"TRAINING EXAMPLE {i + 1}\n")
        f.write("=" * 100 + "\n\n")

        f.write("Instruction:\n")
        f.write(str(example["instruction"]) + "\n\n")

        if example["input"] and str(example["input"]).strip():
            f.write("Medical Question/Context:\n")
            f.write(str(example["input"]) + "\n\n")

        f.write("Answer:\n")
        f.write(str(example["output"]) + "\n\n")

In [15]:
for i in range(5):
    print("=" * 80)
    print(f"EXAMPLE {i + 1}")
    print(train_dataset[i]["text"])

EXAMPLE 1
### Instruction:
Explain why a mass in the lung could cause shortness of breath.

### Medical Question/Context:
<noinput>

### Answer:
A mass in the lung could cause shortness of breath due to several reasons. First, the mass can physically obstruct the air passages, causing difficulty in airflow and leading to breathing difficulties. Second, if the mass is cancerous or infected, it can cause inflammation and damage to lung tissue, reducing its functional capacity and compromising normal breathing. Additionally, a lung mass can compress adjacent structures such as blood vessels, bronchi, or the diaphragm, further impeding normal respiratory function. Overall, any interference with the normal flow of air in the lungs caused by a mass can result in inadequate oxygen exchange and subsequent shortness of breath.

The answer is: A mass in the lung can obstruct air passages, cause inflammation, damage lung tissue, and compress adjacent structures, leading to shortness of breath.
EX

In [16]:
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [17]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [18]:
training_args = TrainingArguments(
    output_dir="./medical_qlora_results",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_steps=100,
    save_total_limit=2,

    fp16=True,

    optim="adamw_8bit",

    report_to="none",

    seed=42,
)

In [19]:
logging_steps=10

In [20]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="MedInstruct-52k.json",
    split="train"
)

print("Dataset loaded successfully!")
print("Total examples:", len(dataset))
print("Columns:", dataset.column_names)

Dataset loaded successfully!
Total examples: 52002
Columns: ['instruction', 'input', 'output']


In [21]:
train_dataset = dataset

print("Total training examples:", len(train_dataset))
print("Training columns:", train_dataset.column_names)

Total training examples: 52002
Training columns: ['instruction', 'input', 'output']


In [22]:
def format_medical_example(example):
    instruction = example["instruction"]
    user_input = example["input"]
    answer = example["output"]

    if user_input and str(user_input).strip():
        text = f"""### Instruction:
{instruction}

### Medical Question/Context:
{user_input}

### Answer:
{answer}"""
    else:
        text = f"""### Instruction:
{instruction}

### Answer:
{answer}"""

    return {"text": text}

In [23]:
medical_data = train_dataset.map(format_medical_example)

print("Total formatted examples:", len(medical_data))

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

Total formatted examples: 52002


In [24]:
training_args = TrainingArguments(
    output_dir="./medical_qlora_results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,

    logging_strategy="steps",
    logging_steps=10,

    save_strategy="steps",
    save_steps=100,

    fp16=True,
    optim="adamw_8bit",
    report_to="none",
)

In [25]:
print("Model:", type(model))
print("Tokenizer:", type(tokenizer))
print("Training examples:", len(train_dataset))

Model: <class 'peft.peft_model.PeftModelForCausalLM'>
Tokenizer: <class 'transformers.tokenization_utils_tokenizers.TokenizersBackend'>
Training examples: 52002


In [26]:
from trl import SFTTrainer

print("SFTTrainer imported successfully!")

SFTTrainer imported successfully!


In [27]:
print("model exists:", "model" in globals())
print("tokenizer exists:", "tokenizer" in globals())
print("train_dataset exists:", "train_dataset" in globals())

model exists: True
tokenizer exists: True
train_dataset exists: True


In [28]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medical_qlora_results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="adamw_8bit",
    report_to="none",
    seed=42,
)

In [29]:
import trl
import transformers

print("TRL version:", trl.__version__)
print("Transformers version:", transformers.__version__)

TRL version: 0.24.0
Transformers version: 5.5.0


In [30]:
def format_medical_example(example):
    instruction = example["instruction"]
    user_input = example["input"]
    answer = example["output"]

    if user_input and str(user_input).strip():
        text = f"""### Instruction:
{instruction}

### Medical Question/Context:
{user_input}

### Answer:
{answer}"""
    else:
        text = f"""### Instruction:
{instruction}

### Answer:
{answer}"""

    return {"text": text}

In [31]:
print(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [32]:
print("Model exists:", "model" in globals())
print("Tokenizer exists:", "tokenizer" in globals())
print("Training arguments exist:", "training_args" in globals())
print("Dataset exists:", "train_dataset" in globals())

Model exists: True
Tokenizer exists: True
Training arguments exist: True
Dataset exists: True


In [33]:
def format_medical_example(example):
    instruction = example["instruction"]
    user_input = example["input"]
    answer = example["output"]

    if user_input and str(user_input).strip():
        text = f"""### Instruction:
{instruction}

### Medical Question/Context:
{user_input}

### Answer:
{answer}"""
    else:
        text = f"""### Instruction:
{instruction}

### Answer:
{answer}"""

    return {"text": text}


medical_data = dataset.map(format_medical_example)

train_dataset = medical_data

print("Columns:", train_dataset.column_names)
print("Total training examples:", len(train_dataset))

Columns: ['instruction', 'input', 'output', 'text']
Total training examples: 52002


In [34]:
for i, example in enumerate(train_dataset):
    print("=" * 80)
    print(f"QUESTION {i + 1}")
    print("=" * 80)

    print("Instruction:")
    print(example["instruction"])

    if example["input"] and str(example["input"]).strip():
        print("\nMedical Question/Context:")
        print(example["input"])

    print("\nAnswer:")
    print(example["output"])

    print()

Streaming output truncated to the last 5000 lines.
Type 1 diabetes is usually diagnosed by conducting several tests, including:

1. Blood Sugar Test: Doctors measure the blood glucose levels to see if they are abnormally high. This is typically done after fasting for at least eight hours or randomly regardless of the time since the last meal.
2. Glycated Hemoglobin (A1C) Test: This test measures the average blood sugar level over the past two to three months. If the A1C level is higher than normal, it may indicate diabetes.
3. Oral Glucose Tolerance Test (OGTT): This test involves measuring blood sugar levels before and two hours after consuming a sugary drink. Higher-than-normal blood sugar levels during the test can indicate diabetes.
4. Antibody Testing: Type 1 diabetes is often associated with the presence of certain autoantibodies. Testing for these antibodies, such as glutamic acid decarboxylase (GAD), insulin autoantibodies (IAAs), or islet cell antibodies (ICAs), can help confi

In [35]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=format_medical_example,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/52002 [00:00<?, ? examples/s]

  Using cached trl-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
Using cached trl-1.14.0-py3-none-any.whl (925 kB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.9.11 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.1 which is incompatible.
unsloth 2026.9.11 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 1.14.0 which is incompatible.
unsloth-zoo 2026.9.7 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, b

In [1]:
import unsloth
import torch
import transformers
import trl
import peft

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Unsloth:", unsloth.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
Transformers: 5.5.0
TRL: 1.14.0
PEFT: 0.21.0
Unsloth: 2026.9.11
CUDA available: True
GPU: Tesla T4


In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print("4-bit model loaded successfully!")

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


4-bit model loaded successfully!


In [12]:
!wget -q https://huggingface.co/datasets/casey-martin/MedInstruct/resolve/main/data/MedInstruct-52k.json -O MedInstruct-52k.json

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="MedInstruct-52k.json",
    split="train"
)

print("Dataset columns:", dataset.column_names)
print("Total examples:", len(dataset))

Generating train split: 0 examples [00:00, ? examples/s]

Dataset columns: ['instruction', 'input', 'output']
Total examples: 52002


In [13]:
def format_medical_example(example):
    instruction = example["instruction"]
    user_input = example["input"]
    answer = example["output"]

    if user_input and str(user_input).strip():
        text = f"""### Instruction:
{instruction}

### Medical Question/Context:
{user_input}

### Answer:
{answer}"""
    else:
        text = f"""### Instruction:
{instruction}

### Answer:
{answer}"""

    return {"text": text}


train_dataset = dataset.map(format_medical_example)

print("Training columns:", train_dataset.column_names)
print("Total training examples:", len(train_dataset))

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

Training columns: ['instruction', 'input', 'output', 'text']
Total training examples: 52002


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./medical_qlora_results",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_strategy="steps",
    logging_steps=10,

    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,

    fp16=True,

    optim="adamw_8bit",

    report_to="none",
    seed=42,
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [15]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print("4-bit base model loaded!")

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


4-bit base model loaded!


In [16]:
print("Training examples:", len(train_dataset))
print("Dataset columns:", train_dataset.column_names)

Training examples: 52002
Dataset columns: ['instruction', 'input', 'output', 'text']


In [17]:
small_train_dataset = train_dataset.select(range(1000))

print("Training examples:", len(small_train_dataset))

Training examples: 1000


In [19]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print("4-bit base model loaded successfully!")

==((====))==  Unsloth 2026.9.11: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


4-bit base model loaded successfully!


In [20]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters attached successfully!")

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


LoRA adapters attached successfully!


In [21]:
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [22]:
small_train_dataset = train_dataset.select(range(1000))

print("Training examples:", len(small_train_dataset))

Training examples: 1000


In [24]:
from trl import SFTTrainer

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=small_train_dataset,
    formatting_func=formatting_func,
)

print("Trainer created successfully!")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Trainer created successfully!


In [25]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 2 | Total steps = 250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.314389
20,1.168178
30,1.056024
40,1.043192
50,1.022305
60,1.039507
70,1.033087
80,1.115870
90,1.049817
100,1.017377


Unsloth: Restored added_tokens_decoder metadata in ./medical_qlora_results/checkpoint-250/tokenizer_config.json.


TrainOutput(global_step=250, training_loss=1.0084906616210938, metrics={'train_runtime': 1084.959, 'train_samples_per_second': 1.843, 'train_steps_per_second': 0.23, 'total_flos': 1.205634238949376e+16, 'train_loss': 1.0084906616210938, 'epoch': 2.0})

In [26]:
logging_steps=10

In [27]:
import pandas as pd

loss_data = []

for log in trainer.state.log_history:
    if "loss" in log:
        loss_data.append({
            "Step": log.get("step"),
            "Epoch": round(log.get("epoch", 0), 2),
            "Training Loss": round(log.get("loss", 0), 4)
        })

loss_df = pd.DataFrame(loss_data)

print("Total loss records:", len(loss_df))
display(loss_df.head(50))

Total loss records: 25


,Step,Epoch,Training Loss
0,10,0.08,1.3144
1,20,0.16,1.1682
2,30,0.24,1.0560
3,40,0.32,1.0432
4,50,0.40,1.0223
5,60,0.48,1.0395
6,70,0.56,1.0331
7,80,0.64,1.1159
8,90,0.72,1.0498
9,100,0.80,1.0174


In [33]:
from google.colab import files

uploaded = files.upload()

Saving .gitignore to .gitignore
Saving .pre-commit-config.yaml to .pre-commit-config.yaml
Saving .ruff.toml to .ruff.toml
Saving app_test.py to app_test.py
Saving chat_history.json to chat_history.json
Saving chat_history.json.txt to chat_history.json.txt
Saving Chatbot.py to Chatbot.py
Saving LICENSE to LICENSE
Saving README.md to README.md
Saving requirements.txt to requirements.txt
Saving requirements-dev.txt to requirements-dev.txt
Saving requirements-offline.txt to requirements-offline.txt


In [35]:
import os

print(os.listdir("/content"))

['.config', '.pre-commit-config.yaml', 'LICENSE', '.gitignore', 'medical_qlora_results', 'unsloth_compiled_cache', 'chat_history.json.txt', 'MedInstruct-52k.json', 'huggingface_tokenizers_cache', 'README.md', 'requirements-dev.txt', 'Chatbot.py', '.ruff.toml', 'all_medical_training_data.txt', 'chat_history.json', 'requirements-offline.txt', 'app_test.py', 'requirements.txt', 'sample_data']


In [2]:
import os

project_path = "/content"

print("Project exists:", os.path.exists(project_path))
print("\nProject files:")
print(os.listdir(project_path))

Project exists: True

Project files:
['.config', 'sample_data']


In [5]:
import os
project_path = "/content" # Set the project path to the correct Colab directory
os.chdir(project_path)

print(os.getcwd())

/content


In [6]:
project_path = r"D:\Downloads\llm-examples-main\llm-examples-main"

In [7]:
project_path = "/content"

In [2]:
import sys
import os

print("Operating System:", sys.platform)
print("Python executable:", sys.executable)
print("Current directory:", os.getcwd())

Operating System: win32
Python executable: C:\Users\sanni\AppData\Local\Programs\Python\Python311\python.exe
Current directory: D:\Downloads\llm-examples-main\llm-examples-main


In [3]:
project_path = r"D:\Downloads\llm-examples-main\llm-examples-main"

print("Project exists:", os.path.exists(project_path))
print("Project path:", project_path)

if os.path.exists(project_path):
    print("\nFiles/folders:")
    for item in os.listdir(project_path):
        print(" -", item)

Project exists: True
Project path: D:\Downloads\llm-examples-main\llm-examples-main

Files/folders:
 - .devcontainer
 - .github
 - .gitignore
 - .pre-commit-config.yaml
 - .ruff.toml
 - .venv
 - app_test.py
 - Chatbot.py
 - chat_history.json
 - chat_history.json.txt
 - LICENSE
 - pages
 - README.md
 - requirements-dev.txt
 - requirements-offline.txt
 - requirements.txt
 - venv


In [4]:
for item in os.listdir(project_path):
    print(item)

.devcontainer
.github
.gitignore
.pre-commit-config.yaml
.ruff.toml
.venv
app_test.py
Chatbot.py
chat_history.json
chat_history.json.txt
LICENSE
pages
README.md
requirements-dev.txt
requirements-offline.txt
requirements.txt
venv


In [6]:
import os

project_path = r"D:\Downloads\llm-examples-main\llm-examples-main"
notebook_path = os.path.join(project_path, "Medical_QLoRA_Unsloth.ipynb")

print("Project exists:", os.path.exists(project_path))
print("Notebook exists:", os.path.exists(notebook_path))
print("Notebook path:", notebook_path)

Project exists: True
Notebook exists: False
Notebook path: D:\Downloads\llm-examples-main\llm-examples-main\Medical_QLoRA_Unsloth.ipynb


In [7]:
import os

project_path = r"D:\Downloads\llm-examples-main\llm-examples-main"

for root, dirs, files in os.walk(project_path):
    for file in files:
        if file.lower() == "medical_qlora_unsloth.ipynb":
            print("FOUND:")
            print(os.path.join(root, file))

In [8]:
import os

for root, dirs, files in os.walk(r"D:\\"):
    for file in files:
        if file.lower() == "medical_qlora_unsloth.ipynb":
            print("FOUND:")
            print(os.path.join(root, file))